In [15]:
from os import system

from anthropic.types import stop_reason
# Install Dependencies
%pip install anthropic python-dotenv

# Setup: load credentials, build the client.
import os
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# find_dotenv(usecwd=True) walks up from the notebook, so .env is found whether it
# lives in ClaudeApi-Notebook/ or the repo root. override=True picks up a rotated key.
load_dotenv(find_dotenv(usecwd=True), override=True)

client = Anthropic()          # reads ANTHROPIC_API_KEY from the environment
model = "claude-haiku-4-5"

k = os.environ["ANTHROPIC_API_KEY"]
print(f"key {k[:13]}...{k[-4:]} (len={len(k)})  |  model {model}")


Note: you may need to restart the kernel to use updated packages.
key sk-ant-api03-...TwAA (len=108)  |  model claude-haiku-4-5


In [16]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


In [17]:
#Make a request
def chat(messages, system=None, temperature=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system
    if temperature is not None:
        params["temperature"] = temperature
    if stop_sequences:
        params["stop_sequences"] = stop_sequences   # NOTE: plural, takes a list

    response = client.messages.create(**params)

    # response.content is a LIST OF BLOCKS (thinking, text, tool_use, ...).
    # Never assume content[0] is text - branch on block.type.
    return "".join(b.text for b in response.content if b.type == "text")


In [18]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MyEventRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [19]:
import json
json.loads(text.strip())


{'Name': 'MyEventRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}